# GDELT Article Body Scraper

This notebook iterates over the URLs in the cleaned GDELT dataset and attempts to 
extract the body text of each article using HTTP requests and BeautifulSoup HTML 
parsing. For each article, irrelevant HTML elements (scripts, navigation, footers) 
are removed and the first three substantive paragraphs are extracted. This body text 
will later be used alongside article titles to compare the quality of sentiment 
signals extracted by FinBERT from headlines only versus headlines plus body text — 
an experiment designed to quantify headline bias in financial sentiment analysis.

**Pipeline:**
1. Scrape all articles → save raw immediately to `01_data/raw/news/gdelt_wti_with_body_raw.csv`
2. Clean characters and filter boilerplate → save to `01_data/processed/gdelt_wti_with_body.csv`

### Necessary imports

In [2]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time
import re
import html

### Read processed news

In [3]:
df_clean = pd.read_csv("../01_data/processed/gdelt_wti_clean.csv", parse_dates=['datetime'])
print(f"Articles loaded: {len(df_clean)}")

Articles loaded: 16326


### Scraping function

Discards irrelevant HTML elements such as `script`, `style`, `nav`, `footer`, `header`, `aside` and extracts the first 3 substantive paragraphs (>50 characters) of each article.

In [4]:
def scrape_article(url, timeout=10):
    headers = {
        "User-Agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
    }
    try:
        response = requests.get(url, headers=headers, timeout=timeout)
        soup = BeautifulSoup(response.text, 'html.parser')

        # Remove irrelevant HTML elements
        for tag in soup(['script', 'style', 'nav', 'footer', 'header', 'aside']):
            tag.decompose()

        # Extract first 3 substantive paragraphs
        paragraphs = [p.get_text(strip=True) for p in soup.find_all('p') if len(p.get_text(strip=True)) > 50]
        body = ' '.join(paragraphs[:3])
        return body

    except Exception as e:
        return f"ERROR: {str(e)[:100]}"

### Test scraping on 5 sample articles

In [6]:
test_urls = df_clean.sample(5, random_state=42)[['title', 'url', 'domain']].reset_index(drop=True)

for _, row in test_urls.iterrows():
    print(f"\nDomain: {row['domain']}")
    print(f"Title: {row['title'][:80]}")
    print(f"URL: {row['url']}")
    body = scrape_article(row['url'])
    print(f"Body ({len(body)} chars): {body[:200]}")
    print("-" * 60)


Domain: kdhnews.com
Title: The Latest : US stock market and global trade partners react to Trump new tariff
URL: https://kdhnews.com/news/nation/the-latest-us-stock-market-and-global-trade-partners-react-to-trump-s-new-tariffs/article_68142557-ebdf-501f-99c9-b8c493d36dc7.html
Body (210 chars): The page may have moved, you may have mistyped the address, or followed a bad link. Visit ourhomepage, or search for whatever you were looking for… Account processing issue - the email address may alr
------------------------------------------------------------

Domain: geopoliticalmonitor.com
Title: Grand Tortue Ahmeyim Field Boosts Senegal and Mauritania Gas Production
URL: https://www.geopoliticalmonitor.com/grand-tortue-ahmeyim-field-boosts-senegal-and-mauritania-gas-production/
Body (2292 chars): Recent developments in offshore hydrocarbon production are set to significantly reshape the economic prospects of two African countries along the continent’s Atlantic coast. On January 6, Senegal a

### Scrape all articles

Iterates over all articles and extracts body text. Progress is printed every 50 articles.

**Warning:** This cell takes 30-60 minutes to complete. Raw results are saved immediately after the loop finishes — do not interrupt before saving.

In [7]:
bodies = []
total = len(df_clean)

for i, row in df_clean.iterrows():
    body = scrape_article(row['url'])
    bodies.append(body)

    if (i + 1) % 50 == 0:
        print(f"{i + 1}/{total} — {row['domain']}")

    time.sleep(0.5)

df_clean['body'] = bodies

# Save raw immediately before any filtering or cleaning
filename = "gdelt_wti_with_body_raw.csv"
df_clean.to_csv(f"../01_data/raw/news/{filename}", index=False)
print(f"\nRaw body data saved to 01_data/raw/news/{filename}")

# Summary
successful = df_clean[~df_clean['body'].str.startswith('ERROR') & (df_clean['body'].str.len() > 100)]
print(f"Total articles: {total}")
print(f"With body extracted: {len(successful)} ({len(successful)/total*100:.1f}%)")
print(f"Failed or empty: {total - len(successful)}")

50/16326 — zerohedge.com
100/16326 — hellenicshippingnews.com
150/16326 — arabnews.com
200/16326 — marketscreener.com
250/16326 — bnnbloomberg.ca
300/16326 — aninews.in
350/16326 — fool.com
400/16326 — peoplesdailyng.com
450/16326 — seekingalpha.com
500/16326 — jpost.com
550/16326 — brecorder.com
600/16326 — kuna.net.kw
650/16326 — seekingalpha.com
700/16326 — zerohedge.com
750/16326 — kbtx.com
800/16326 — energy.einnews.com
850/16326 — themarketsdaily.com
900/16326 — themarketsdaily.com
950/16326 — marketscreener.com
1000/16326 — morningstar.com
1050/16326 — polity.org.za
1100/16326 — india.com
1150/16326 — brecorder.com
1200/16326 — financialexpress.com
1250/16326 — forexlive.com
1300/16326 — forbes.com
1350/16326 — finance.yahoo.com
1400/16326 — brecorder.com
1450/16326 — businesstoday.in
1500/16326 — thepeninsulaqatar.com
1550/16326 — globenewswire.com
1600/16326 — prokerala.com
1650/16326 — usatoday.com
1700/16326 — fxstreet.com
1750/16326 — seekingalpha.com
1800/16326 — prnewswir

### Diagnostic — check for dirty characters

Reads from the raw CSV to check for HTML entities, newlines, and control characters that could corrupt downstream processing.

In [8]:
df_raw = pd.read_csv("../01_data/raw/news/gdelt_wti_with_body_raw.csv")

issues = []
for i, row in df_raw.iterrows():
    body = str(row['body'])
    if '&amp;' in body or '&nbsp;' in body or '&#' in body:
        issues.append((i, 'HTML entities', body[:100]))
    if '\n' in body or '\r' in body:
        issues.append((i, 'Newlines', body[:100]))
    if re.search(r'[\x00-\x08\x0b\x0c\x0e-\x1f\x7f]', body):
        issues.append((i, 'Control characters', body[:100]))

print(f"Articles with issues: {len(issues)}")
for idx, issue_type, sample in issues[:10]:
    print(f"\n[{issue_type}] Row {idx}: {sample}")

Articles with issues: 2230

[Newlines] Row 3: Published on 01/01/2024
                    at 12:15 am EST        - Modified on 01/01/2024
        

[Newlines] Row 33: Published on 01/02/2024
                    at 01:00 pm EST        - Modified on 01/02/2024
        

[Newlines] Row 36: Representational image only.
                                          | Photo Credit: The Hindu The

[Newlines] Row 45: Legal Disclaimer:MENAFNprovides the
              information âas isâ without warranty of any ki

[Newlines] Row 50: Legal Disclaimer:MENAFNprovides the
              information âas isâ without warranty of any ki

[Newlines] Row 59: Published on 01/03/2024
                    at 03:43 pm EST HOUSTON, Jan 3 (Reuters) - Venezuela's o

[Newlines] Row 66: Discover the latestBusiness News,Sensex, andNiftyupdates. ObtainPersonal Financeinsights, tax querie

[Newlines] Row 71: Concerns over movement of merchant vessels in the Red Sea have also lifted the price of the commodit

[Newl

### Clean and filter

Reads from the raw CSV and applies two transformations:

- **Character cleaning:** Unescapes HTML entities, removes control characters, normalizes whitespace.
- **Boilerplate filtering:** Allow-list approach where the news are inspected for specific words that potentially are from a legit news instead of boilerplate, errors, empty strings, etc.

In [9]:
"""Character cleaning"""
df_raw = pd.read_csv("../01_data/raw/news/gdelt_wti_with_body_raw.csv")

# --- Character cleaning ---
def clean_body(text):
    if not isinstance(text, str):
        return ''
    text = html.unescape(text)
    try:
        text = text.encode('latin-1').decode('utf-8')
    except (UnicodeDecodeError, UnicodeEncodeError):
        pass
    text = re.sub(r'[\x00-\x08\x0b\x0c\x0e-\x1f\x7f]', '', text)
    text = text.replace('\n', ' ').replace('\r', ' ')
    text = re.sub(r'\s+', ' ', text).strip()
    return text

df_raw['body'] = df_raw['body'].apply(clean_body)
print("Character cleaning done.")

Character cleaning done.


In [16]:
"""Boilerplate allow-listing"""

# Keywords that should appear in legitimate WTI-related financial content
ALLOW_KEYWORDS = [
    'oil', 'crude', 'wti', 'brent', 'opec', 'barrel', 'petroleum',
    'energy', 'price', 'market', 'supply', 'demand', 'production',
    'refinery', 'inventory', 'futures', 'trading', 'commodity',
    'inflation', 'fed', 'dollar', 'interest rate', 'gdp'
]

press_release_phrases = [
    "newsfile corp",
    "tsxv:",
    "otcqx:",
    "tsx:",
    "nyse:",
    "non-ifrs",
    "international financial reporting standards",
    "this press release",
    "forward-looking statements",
    "is pleased to announce",
    "is pleased to provide",
]

invalid_news = []

def is_valid_body(text):
    if not isinstance(text, str):
        return False
    if len(text) < 400:
        return False
    if text.startswith('ERROR'):
        return False
    
    text_lower = text.lower()
    
    # Must contain at least one energy/financial keyword
    has_keyword = any(kw in text_lower for kw in ALLOW_KEYWORDS)
    if not has_keyword:
        invalid_news.append(text)
        return False
    if any(phrase in text_lower for phrase in press_release_phrases):
        return False
    
    # Reject if more than 30% of text is capitalized words (navigation/boilerplate pattern)
    words = text.split()
    if len(words) > 10:
        cap_ratio = sum(1 for w in words if w.isupper() and len(w) > 2) / len(words)
        if cap_ratio > 0.3:
            return False
    
    return True

df_raw['body_valid'] = df_raw['body'].apply(is_valid_body)
print(f"Valid bodies: {df_raw['body_valid'].sum()}")
print(f"Discarded: {(~df_raw['body_valid']).sum()}")
print(df_raw['body'].str.len().describe().round(0))

# Save processed
filename = "gdelt_wti_with_body.csv"
df_raw.to_csv(f"../01_data/processed/{filename}", index=False)
print(f"\nSaved to 01_data/processed/{filename}")

Valid bodies: 9068
Discarded: 7258
count     16326.0
mean        773.0
std        2489.0
min           0.0
25%         211.0
50%         509.0
75%         757.0
max      182038.0
Name: body, dtype: float64

Saved to 01_data/processed/gdelt_wti_with_body.csv


### Migrate into db
Data isnow saved into a db, so migration is needed.

In [2]:
import sqlite3
import pandas as pd

conn = sqlite3.connect("../01_data/wti_thesis.db")

# Read the processed CSV with bodies
df = pd.read_csv("../01_data/processed/gdelt_wti_with_body.csv", parse_dates=['datetime'])

# Add explicit ID
df = df.reset_index(drop=True)
df['id'] = df.index + 1
df['datetime'] = df['datetime'].astype(str)

# Save to articles table
articles_cols = ['id', 'title', 'datetime', 'domain', 'url', 'body', 'body_valid']
df[articles_cols].to_sql('articles', conn, if_exists='replace', index=False)

print(f"Articles migrated: {len(df)}")
print(pd.read_sql("SELECT COUNT(*) as total FROM articles", conn))

conn.close()

Articles migrated: 16326
   total
0  16326
